In [ ]:
"""
Pipeline de Consolidação - Consolidação Trimestral: Frutally Nordeste Q1/2025
"""

import pandas as pd
import json
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
log = logging.getLogger(__name__)

In [89]:
# =====================================================================
# FUNÇÕES AUXILIARES
# =====================================================================

def converter_datas(serie: pd.Series) -> pd.Series:
    """Converte datas nos formatos ISO (YYYY-MM-DD) e BR (DD/MM/YYYY) para um único formato"""
    iso = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')
    br = pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')

    resultado = iso.fillna(br)

    nao_parseadas = resultado.isna().sum()
    if nao_parseadas > 0:
        log.warning(f'{nao_parseadas} datas não puderam ser parseadas.')

    return resultado

def converter_moeda_br(serie: pd.Series) -> pd.Series:
    """Converte valores em reais (R$ 1.250,00) para o formato do Pandas (1250.00) e converte para float"""
    return (serie
            .str.replace(r'R\$\s*', '', regex=True)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .astype(float))

def converter_percentual(serie: pd.Series) -> pd.Series:
    """Converte percentuais (%) para decimal"""
    return ((serie
 .str.replace('%', '', regex=False)
 .str.replace(',', '.', regex=False)
 .str.strip()
 .astype(float)
 .div(100)))

def limpar_colunas_string(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Remove espaços duplos em colunas que são strings e aplica strip()"""
    for col in colunas:
        df[col] = (df[col]
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    
    return df


In [ ]:
# =====================================================================
# ETAPA 1 — LEITURA DOS DADOS
# =====================================================================

vendas_jan2025 = pd.read_csv('vendas_jan2025.csv', sep=';', encoding='latin-1')
log.info(f'vendas_jan2025: {vendas_jan2025.shape[0]} registros lidos.')

with open('vendas_fev2025.json', 'r', encoding='utf-8') as f:
    vendas_fev2025 = pd.json_normalize(json.load(f), sep='_')
log.info(f'vendas_fev2025: {vendas_fev2025[0]} registros lidos.')

vendas_mar2025 = pd.read_excel('vendas_mar2025.xlsx', sheet_name='dados')
log.info(f'vendas_mar2025 (dados): {vendas_mar2025.shape[0]} registros lidos.')

lojas_nordeste = pd.read_excel('lojas_nordeste.xlsx', sheet_name='ativas')
log.info(f'lojas_nordeste (ativas): {lojas_nordeste.shape[0]} registros lidos.')

metas_q1_2025 = pd.read_csv('metas_q1_2025.csv', sep=',', encoding='utf-8')
log.info(f'metas_q1_2025: {metas_q1_2025.shape[0]} registros lidos.')

### Limpeza dos arquivos
# CONTINUAR DAQUI

In [91]:
vendas_jan2025 = vendas_jan2025.drop_duplicates()

vendas_jan2025['data_venda'] = converter_datas(vendas_jan2025['data_venda'])
vendas_jan2025['preco_unitario'] = converter_moeda_br(vendas_jan2025['preco_unitario'])
vendas_jan2025['desconto_percentual'] = converter_percentual(vendas_jan2025['desconto_percentual'])

colunas_string = vendas_jan2025.select_dtypes('string').columns
vendas_jan2025 = limpar_colunas_string(vendas_jan2025, colunas_string)
vendas_jan2025['vendedor'] = vendas_jan2025['vendedor'].str.title()

vendas_jan2025 = vendas_jan2025.reset_index(drop=True)


In [92]:
vendas_fev2025 = vendas_fev2025.drop_duplicates()

vendas_fev2025['data_venda'] = pd.to_datetime(vendas_fev2025['data_venda'], format='%d-%m-%Y')
vendas_fev2025['desconto_percentual'] = converter_percentual(vendas_fev2025['desconto_percentual'])

colunas_string = vendas_fev2025.select_dtypes('str').columns
vendas_fev2025 = limpar_colunas_string(vendas_fev2025, colunas_string)
vendas_fev2025['vendedor'] = vendas_fev2025['vendedor'].str.title()

vendas_fev2025 = vendas_fev2025.reset_index(drop=True)

In [93]:
vendas_mar2025 = vendas_mar2025.drop_duplicates()

vendas_mar2025['preco_unitario'] = converter_moeda_br(vendas_mar2025['preco_unitario'])
vendas_mar2025['desconto_percentual'] = (vendas_mar2025['desconto_percentual'].div(100))

colunas_strings = vendas_mar2025.select_dtypes('str').columns
vendas_mar2025 = limpar_colunas_string(vendas_mar2025, colunas_strings)
vendas_mar2025['vendedor'] = vendas_mar2025['vendedor'].str.title()

vendas_mar2025 = vendas_mar2025.reset_index(drop=True)

In [94]:
lojas_nordeste = lojas_nordeste.drop_duplicates()

lojas_nordeste['data_inauguracao'] = converter_datas(lojas_nordeste['data_inauguracao'])

colunas_strings = lojas_nordeste.select_dtypes('str').columns
lojas_nordeste = limpar_colunas_string(lojas_nordeste, colunas_strings)

lojas_nordeste = lojas_nordeste.reset_index(drop=True)

In [95]:
metas_q1_2025 = metas_q1_2025.drop_duplicates()

metas_q1_2025['meta_faturamento_q1'] = converter_moeda_br(metas_q1_2025['meta_faturamento_q1'])

metas_q1_2025 = metas_q1_2025.reset_index(drop=True)

### Merge dos Dataframes

In [96]:
vendas_trimestre = pd.concat([vendas_jan2025, vendas_fev2025, vendas_mar2025], axis=0, ignore_index=True, verify_integrity=True)

vendas_trimestre = (vendas_trimestre.merge(right=lojas_nordeste, how='inner', on='id_loja')).merge(right=metas_q1_2025, how='inner', on='id_loja')

### Coluna Calculada

In [97]:
vendas_trimestre['valor_total_venda'] = (vendas_trimestre['quantidade'] * 
                                         vendas_trimestre['preco_unitario'] * 
                                         (1 - vendas_trimestre['desconto_percentual'])
                                         ).round(2)

### Limpeza final

In [98]:
vendas_trimestre = vendas_trimestre.dropna()

vendas_trimestre = vendas_trimestre.reset_index(drop=True)

### Exportação

In [99]:
vendas_trimestre.to_csv('relatorio_frutally_jan2025.csv', sep=',', index=False, encoding='utf-8')